# Dataset Curation for StyleTTS2
> This is a private/commercial notebook built by Trelis Research. Purchase a copy at Trelis.com.

This Jupyter notebook guides you through the process of creating an audio dataset from YouTube videos or from your own audio dataset. It covers downloading audio, converting formats, transcribing, segmenting, and preparing the data for upload to Hugging Face Hub.

Subscribe to Trelis Research emails here and get notified each time a new video tutorial is published.

Attribution:
This notebook builds on scripts from the following repos:
- [WhisperX](https://github.com/VR-13/WhisperX).
- [StyleTTS2](https://github.com/yl4579/StyleTTS2).

## Choosing a GPU

> Colab is highly recommended for this specific script because Youtube tends to block downloads to other sites!

You have a few options:
1. Hosted service, e.g. Runpod:
- Runpod one-click template [here](https://runpod.io/gsc?template=ifyqsvjlzj&ref=jmfkcdio) - easier setup.
2. Google Colab (fine for 7B models or smaller):
- Upload the .ipynb notebook
- Select a T4 GPU from Runtime -> Change Runtime Type.
3. Your own computer (assuming you have an AMD or Nvidia GPU) - ADVANCED:
- Set up jupyter lab and a virtual environment using the instructions [here](https://github.com/TrelisResearch/install-guides/blob/main/jupyter-lab-setup.md)

## Setup and Dependencies
First, we need to install all the required dependencies. Run the following cell to install the necessary packages and tools.

In [2]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -qU
!apt-get update -qq > /dev/null # update quietly
!apt-get install espeak-ng -y > /dev/null # install quietly
!pip install git+https://github.com/m-bain/whisperx.git  -q # you may be required to re-start the runtime during installation.
!pip install pydub -qU
!pip install pytube -qU
!pip install tqdm -q
# !git clone https://github.com/rs545837/StyleTTS2FineTune.git
!apt-get install ffmpeg -y > /dev/null # install quietly

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.5/808.5 kB 34.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Creating all the required directories
srtsegmenter.py file creates the directories needed to process all the audio wav files.

In [3]:
# if you need to restart
!rm -r paddedAudio segmentedAudio offLengthAudio trainingdata

rm: cannot remove 'paddedAudio': No such file or directory
rm: cannot remove 'segmentedAudio': No such file or directory
rm: cannot remove 'offLengthAudio': No such file or directory
rm: cannot remove 'trainingdata': No such file or directory


In [13]:
# Create directories

import os
import glob

output_dir = './segmentedAudio/'   # path to where you want the segmented audio to go.
off_length_audio_dir = './offLengthAudio/' # path to where you want the bad audio to go.
srt_dir = './srt/'
audio_dir = './audio/'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(off_length_audio_dir, exist_ok=True) # only required if doing simple segmentation.
os.makedirs(srt_dir, exist_ok=True)
os.makedirs(audio_dir, exist_ok=True)
os.makedirs('./trainingdata', exist_ok=True)

srt_list = glob.glob("./srt/*.srt") # Gets a list of all srt files
audio_list = glob.glob("./audio/*.wav") # Gets a list of all audio files

## Downloading Audio from YouTube
This section allows you to download audio from YouTube videos and convert the audio to wav format. Replace the youtube_link variable with the URL of the video you want to use.

### Download an audio wav file from youtube video link and store it in audio folder

In [5]:
!pip install yt_dlp -qU

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.


In [6]:
from yt_dlp import YoutubeDL

def download_audio(youtube_link, output_path="audio"):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': f'{output_path}/%(title)s.%(ext)s'
    }
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_link])
    print(f"Audio downloaded from {youtube_link} and saved to {output_path}")

# Example Usage
youtube_link = "https://youtu.be/lJDxkjE9SSY" # Trelis Research
download_audio(youtube_link)

[youtube] Extracting URL: https://youtu.be/lJDxkjE9SSY
[youtube] lJDxkjE9SSY: Downloading webpage
[youtube] lJDxkjE9SSY: Downloading ios player API JSON
[youtube] lJDxkjE9SSY: Downloading player 820bff3b
[youtube] lJDxkjE9SSY: Downloading m3u8 information
[info] lJDxkjE9SSY: Downloading 1 format(s): 251
[download] Destination: audio/Top Ten Fine Tuning Tips.webm
[download] 100% of   23.54MiB in 00:00:01 at 16.10MiB/s  
[ExtractAudio] Destination: audio/Top Ten Fine Tuning Tips.mp3
Deleting original file audio/Top Ten Fine Tuning Tips.webm (pass -k to keep)
Audio downloaded from https://youtu.be/lJDxkjE9SSY and saved to audio


### Convert from mp4 Audio to WAV

In [7]:
import os
from pydub import AudioSegment

def convert_to_wav(input_file, output_file):
    audio = AudioSegment.from_file(
        input_file,
        # format="mp4"
        format="mp3"
        )
    audio.export(output_file, format="wav")

# Get the mp4 files
audio_folder = "audio"
# input_files = [f for f in os.listdir(audio_folder) if f.endswith(".mp4")]
input_files = [f for f in os.listdir(audio_folder) if f.endswith(".mp3")]

if input_files:
    for input_file in input_files:
        input_path = os.path.join(audio_folder, input_file)

        # Ask for the new filename
        new_name = input(f"Enter new name for {input_file} (without extension): ")
        output_path = os.path.join(audio_folder, new_name + ".wav")

        # Convert to wav
        convert_to_wav(input_path, output_path)
        print(f"Converted {input_file} to {new_name}.wav")

        # Delete the original mp4 file
        os.remove(input_path)
        print(f"Deleted {input_file}")

    print("All conversions completed.")
else:
    print("No audio input files found in the audio folder.")

Enter new name for Top Ten Fine Tuning Tips.mp3 (without extension): toptentips-text
Converted Top Ten Fine Tuning Tips.mp3 to toptentips-text.wav
Deleted Top Ten Fine Tuning Tips.mp3
All conversions completed.


## Transcribing Audio with WhisperX
Now we'll use WhisperX to transcribe the audio files and generate SRT files.

WhisperX improves on Whisper by:
1. Slicing out silence in the input WAV file (reduces noise).
1. Using a phoneme model to get word level time-stamps. (One example of a phoneme is the "p" sound).

Note that [WAV2VEC2_ASR_LARGE_LV60K_960H](https://pytorch.org/audio/main/generated/torchaudio.pipelines.WAV2VEC2_ASR_LARGE_LV60K_960H.html) is the model being used for the phoneme detection.

In [8]:
!for i in ./audio/*.wav; do whisperx "$i" --model large-v2 --align_model WAV2VEC2_ASR_LARGE_LV60K_960H; done
!mv *.srt ./srt

2024-07-16 12:03:57.508425: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-16 12:03:57.508485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-16 12:03:57.622271: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-16 12:03:59.920355: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/local/lib/python3.10/dist-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set

## Processing SRT Files
This step segments the audio based on the SRT files and adds padding.

### Segment Audio

This script chunks the audio based on the captions.
Notably, it does "buffering", which means that an
overlap is used when creating segments. This prevents
speech being cut off. Specifically:
- if two caption segments have a short gap, use half of
that gap to buffer
- if two caption segments have a long gap, use a
standard buffer of 200 ms.

Note that a speaker id is written to the end of each output file line (e.g. 1\n).

In [9]:
!apt-get update -qq > /dev/null # update quietly
!apt-get install espeak-ng -y > /dev/null # install quietly

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [11]:
from transformers import BertTokenizer
import nltk

# Ensure NLTK's tokenizer models are downloaded
nltk.download('punkt')

# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class TextCleaner:
    def __init__(self):
        # No need for a custom dictionary here as we are using BERT's tokenizer
        pass

    def __call__(self, text):
        # Tokenize using BERT tokenizer
        tokens = tokenizer.tokenize(text)
        # Convert tokens to their corresponding IDs
        token_indices = tokenizer.convert_tokens_to_ids(tokens)
        return token_indices, tokens

# Initialize TextCleaner instance
textcleaner = TextCleaner()

# Define the function to count tokens in a string of text
def count_tokens(text, print_tokens=False):
    # Strip whitespace from the input text
    text = text.strip()

    # Clean and tokenize the text
    token_indices, tokens = textcleaner(text)

    # Insert a starting token (0) at the beginning
    token_indices.insert(0, tokenizer.cls_token_id)  # Use CLS token ID

    if print_tokens:
        print("Token indices: ", token_indices)
        print("Tokens: ", tokens)

    # Return the number of tokens
    return len(token_indices)

# Example usage
text = "Hello, world!"
num_tokens = count_tokens(text, print_tokens=True)
print(f"Number of tokens: {num_tokens}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Token indices:  [101, 7592, 1010, 2088, 999]
Tokens:  ['hello', ',', 'world', '!']
Number of tokens: 5


In [14]:
# DYNAMICALLY COMBINE MULTIPLE SRT ROWS TO GET CLIPS OF A TARGET TOKEN LENGTH
## RECOMMENDED

max_buffer_time = 200  # Max buffer to add to start and end of audio segments in milliseconds

max_token_length = 512 # about 45s clips. 2x batches will fit on an A6000 if using LoRA.
# max_token_length = 200 # about 70s clips. 2x batches will fit on an A100 if using LoRA.

min_token_length = int(max_token_length*0.8) #ensure no rows of data are less than 80% of the max.

import os
import pysrt
from pydub import AudioSegment
from tqdm import tqdm
import logging

# Ensure there are SRT and audio files
if len(srt_list) == 0 or len(audio_list) == 0:
    raise Exception(f"You need to have at least 1 srt file and 1 audio file, you have {len(srt_list)} srt and {len(audio_list)} audio files!")

for sub_file in tqdm(srt_list):  # Iterate over all SRT files
    subs = pysrt.open(sub_file)
    audio_name = os.path.basename(sub_file).replace(".srt", ".wav")

    audio = AudioSegment.from_wav(f'{audio_dir}/{audio_name}')

    with open('./trainingdata/output.txt', 'a+') as out_file:
        i = 0
        save_counter = 0  # Counter for saving files with continuous names
        offlength_counter = 0 # Counter for off-length files

        while i < len(subs):
            start_time = (subs[i].start.minutes * 60 + subs[i].start.seconds) * 1000 + subs[i].start.milliseconds
            combined_end_time = (subs[i].end.minutes * 60 + subs[i].end.seconds) * 1000 + subs[i].end.milliseconds
            combined_text = subs[i].text
            total_tokens = count_tokens(combined_text)

            # Add buffer to the start time for all but the first clip
            if i > 0:
                prev_end_time = (subs[i - 1].end.minutes * 60 + subs[i - 1].end.seconds) * 1000 + subs[i - 1].end.milliseconds
                gap_to_prev = max(0, start_time - prev_end_time)
                start_time = max(0, start_time - min(max_buffer_time, gap_to_prev // 2))

            j = i + 1
            while j < len(subs):
                next_end_time = (subs[j].end.minutes * 60 + subs[j].end.seconds) * 1000 + subs[j].end.milliseconds
                next_duration = next_end_time - start_time
                next_text = combined_text + " " + subs[j].text
                next_token_count = count_tokens(next_text)

                if next_token_count > max_token_length:
                    break
                elif min_token_length <= next_token_count <= max_token_length:
                    combined_end_time = next_end_time
                    combined_text = next_text
                    total_tokens = next_token_count
                    j += 1
                    break
                else:
                    combined_end_time = next_end_time
                    combined_text = next_text
                    total_tokens = next_token_count
                    j += 1

            if j < len(subs):
                next_start_time = (subs[j].start.minutes * 60 + subs[j].start.seconds) * 1000 + subs[j].start.milliseconds
                gap_to_next = max(0, next_start_time - combined_end_time)
                adjustment = min(max_buffer_time, gap_to_next // 2)
                combined_end_time += adjustment
                # print(f"Adjusted end_time for segment {i} by {adjustment}ms: New end_time = {combined_end_time}")
            else:
                combined_end_time += max_buffer_time
                # print(f"Added buffer time to the last segment {i}: New end_time = {combined_end_time}")

            audio_segment = audio[start_time:combined_end_time]

            if min_token_length <= total_tokens <= max_token_length:
                filename = f'{audio_name[:-4]}_{save_counter}.wav'  # Use save_counter for continuous naming
                audio_segment.export(os.path.join(output_dir, filename), format='wav')
                out_file.write(f'{filename}|{combined_text}|1\n')

                # Print out the filename and token length of each audio snippet created
                print(f"{filename}: Token length = {total_tokens}")
                save_counter += 1  # Increment the save counter only if the file is saved
            else:
                filename = f'{audio_name[:-4]}_{offlength_counter}.wav'  # Use save_counter for continuous naming
                audio_segment.export(os.path.join(off_length_audio_dir, filename), format='wav')
                offlength_counter += 1

            i = j

  0%|          | 0/1 [00:00<?, ?it/s]

toptentips-text_0.wav: Token length = 413
toptentips-text_1.wav: Token length = 410
toptentips-text_2.wav: Token length = 430
toptentips-text_3.wav: Token length = 412
toptentips-text_4.wav: Token length = 446
toptentips-text_5.wav: Token length = 418
toptentips-text_6.wav: Token length = 427
toptentips-text_7.wav: Token length = 411
toptentips-text_8.wav: Token length = 416
toptentips-text_9.wav: Token length = 409
toptentips-text_10.wav: Token length = 457
toptentips-text_11.wav: Token length = 418


100%|██████████| 1/1 [00:03<00:00,  3.52s/it]

toptentips-text_12.wav: Token length = 418


#### Other audio segmentation methods

In [ ]:
# # SIMPLE/NAIVE APPROACH - ONE AUDIO SEGMENT PER LINE OF THE SRT FILE

# import os
# import pysrt
# from pydub import AudioSegment
# from tqdm import tqdm

# if len(srt_list) == 0 or len(audio_list) == 0:
#     raise Exception(f"You need to have at least 1 srt file and 1 audio file, you have {len(srt_list)} srt and {len(audio_list)} audio files!")

# for sub_file in tqdm(srt_list):  # Iterate over all srt files
#     subs = pysrt.open(sub_file)
#     audio_name = os.path.basename(sub_file).replace(".srt", ".wav")

#     audio = AudioSegment.from_wav(f'./{audio_dir}/{audio_name}')

#     max_buffer_time = 200  # Max buffer to add to start and end of audio segments

#     with open('./trainingdata/output.txt', 'a+') as out_file:
#         for i, sub in enumerate(subs):
#             # get start and end times in milliseconds
#             start_time = (sub.start.minutes * 60 + sub.start.seconds) * 1000 + sub.start.milliseconds
#             end_time = (sub.end.minutes * 60 + sub.end.seconds) * 1000 + sub.end.milliseconds

#             # Add buffer to the start time
#             if i == 0:
#                 # For the first clip, add the full max buffer time to the start
#                 start_time = max(0, start_time - max_buffer_time)
#             else:
#                 # For other clips, add the min of half the gap to the previous clip and max buffer time
#                 prev_sub = subs[i - 1]
#                 prev_end_time = (prev_sub.end.minutes * 60 + prev_sub.end.seconds) * 1000 + prev_sub.end.milliseconds
#                 if start_time - prev_end_time < 0:
#                     print("Negative gap between captions, check the input srt file.")
#                 gap_to_prev = max(0, start_time - prev_end_time)
#                 start_time = max(0, start_time - min(max_buffer_time, gap_to_prev // 2))

#             if i < len(subs) - 1:
#                 next_sub = subs[i + 1]
#                 next_start_time = (next_sub.start.minutes * 60 + next_sub.start.seconds) * 1000 + next_sub.start.milliseconds
#                 if next_start_time - end_time < 0:
#                     print("Negative gap between captions, check the input srt file.")
#                 # get the gap to the next subtitle
#                 gap_to_next = max(0, next_start_time - end_time)

#                 # Add a buffer, but don't add more than half of the gap (we don't want to include any of the next phrase)
#                 adjustment = min(max_buffer_time, gap_to_next // 2)
#                 end_time += adjustment
#                 print(f"Adjusted end_time for segment {i} by {adjustment}ms: New end_time = {end_time}")
#             else:
#                 # If this is the last subtitle, add the buffer time to the end time
#                 end_time += max_buffer_time
#                 print(f"Added buffer time to the last segment {i}: New end_time = {end_time}")

#             audio_segment = audio[start_time:end_time]
#             duration = len(audio_segment)
#             filename = f'{audio_name[:-4]}_{i}.wav'

#             # If the duration is within the desired range, then we keep it, anything outside this range goes into the offLengthAudio folder
#             if 1850 <= duration <= 12000:   # Adjust these values to your desired range
#                 audio_segment.export(os.path.join(output_dir, filename), format='wav')
#                 out_file.write(f'{filename}|{sub.text}|1\n')
#             else:
#                 audio_segment.export(os.path.join(off_length_audio_dir, filename), format='wav')


In [ ]:
# # DYNAMICALLY COMBINE MULTIPLE SRT ROWS TO GET CLIPS OF A TARGET LENGTH
# # Creates data rows within a specific audio duration range.
# # NOT RECOMMENDED: It's better to create data by phoneme length because ultimately the BERT encoder is the limitation on the overall model context.

# import os
# import pysrt
# from pydub import AudioSegment
# from tqdm import tqdm

# if len(srt_list) == 0 or len(audio_list) == 0:
#     raise Exception(f"You need to have at least 1 srt file and 1 audio file, you have {len(srt_list)} srt and {len(audio_list)} audio files!")

# for sub_file in tqdm(srt_list):  # Iterate over all srt files
#     subs = pysrt.open(sub_file)
#     audio_name = os.path.basename(sub_file).replace(".srt", ".wav")

#     audio = AudioSegment.from_wav(f'./{audio_dir}/{audio_name}')

#     max_buffer_time = 200  # Max buffer to add to start and end of audio segments
#     min_duration = 20000 # ms
#     max_duration = 30000 # ms

#     with open('./trainingdata/output.txt', 'a+') as out_file:
#         i = 0
#         save_counter = 0  # Counter for saving files with continuous names

#         while i < len(subs):
#             start_time = (subs[i].start.minutes * 60 + subs[i].start.seconds) * 1000 + subs[i].start.milliseconds
#             combined_end_time = (subs[i].end.minutes * 60 + subs[i].end.seconds) * 1000 + subs[i].end.milliseconds
#             combined_text = subs[i].text

#             # Add buffer to the start time for all but the first clip
#             if i > 0:
#                 prev_end_time = (subs[i - 1].end.minutes * 60 + subs[i - 1].end.seconds) * 1000 + subs[i - 1].end.milliseconds
#                 gap_to_prev = max(0, start_time - prev_end_time)
#                 start_time = max(0, start_time - min(max_buffer_time, gap_to_prev // 2))

#             j = i + 1
#             while j < len(subs):
#                 next_end_time = (subs[j].end.minutes * 60 + subs[j].end.seconds) * 1000 + subs[j].end.milliseconds
#                 next_duration = next_end_time - start_time

#                 if next_duration > max_duration:
#                     break
#                 elif min_duration <= next_duration <= max_duration:
#                     combined_end_time = next_end_time
#                     combined_text += " " + subs[j].text
#                     j += 1
#                     break
#                 else:
#                     combined_end_time = next_end_time
#                     combined_text += " " + subs[j].text
#                     j += 1

#             if j < len(subs):
#                 next_start_time = (subs[j].start.minutes * 60 + subs[j].start.seconds) * 1000 + subs[j].start.milliseconds
#                 gap_to_next = max(0, next_start_time - combined_end_time)
#                 adjustment = min(max_buffer_time, gap_to_next // 2)
#                 combined_end_time += adjustment
#                 print(f"Adjusted end_time for segment {i} by {adjustment}ms: New end_time = {combined_end_time}")
#             else:
#                 combined_end_time += max_buffer_time
#                 print(f"Added buffer time to the last segment {i}: New end_time = {combined_end_time}")

#             audio_segment = audio[start_time:combined_end_time]
#             duration = len(audio_segment)
#             filename = f'{audio_name[:-4]}_{save_counter}.wav'  # Use save_counter for continuous naming

#             if min_duration <= duration <= max_duration:
#                 audio_segment.export(os.path.join(output_dir, filename), format='wav')
#                 out_file.write(f'{filename}|{combined_text}|1\n')
#                 save_counter += 1  # Increment the save counter only if the file is saved
#             else:
#                 audio_segment.export(os.path.join(off_length_audio_dir, filename), format='wav')

#             i = j

### Add Padding
Optionally, you can pad the start and end of your audio segments with silence.

This may not be necessary because we have already added buffering at the start and end of each segment.

In [15]:
# IF USING THIS, THERE'S ALSO THE QUESTION OF WHETHER THE OUTPUT SRT TIMESTAMPS SHOULD BE UPDATED AS WELL BY THE PADDING.

from pydub import AudioSegment
import os
import glob

padding = 0 # ms. Default is to add none, but do run the script to create the "padded" files

# input path to segmentedAudio folder here and create the paddedAudio folder and put its path here as well.
source_dir = './segmentedAudio'
target_dir = './paddedAudio'

os.makedirs(target_dir, exist_ok=True)

wav_files = glob.glob(os.path.join(source_dir, '*.wav'))

for wav_file in wav_files:
    audio = AudioSegment.from_wav(wav_file)

    # duration of silence in ms
    silence = AudioSegment.silent(duration=padding) # This is length of silence to pad beginning and end of segments. You can change this to whatever you want.
    new_audio = silence + audio + silence
    new_file_path = os.path.join(target_dir, os.path.basename(wav_file))
    new_audio.export(new_file_path, format='wav')

print("Processing complete. Modified files are saved in:", target_dir)

Processing complete. Modified files are saved in: ./paddedAudio


### Listen to audio clips
Listen to different clips by incrementing the number "_0".

You should be aiming for:
- Clips that are just shorter than the max. length allowed by the model you will train. (Longer length = higher quality). Max token length is 512 for StyleTTS2.
- Clips that include full sentences OR multiple sentences.
- Appropriate pauses between phrases in the same clip.
- Appropriate pauses at the start and end of clips.

In [17]:
import os
from IPython.display import Audio, display

# Define the directory containing the audio files
audio_dir = '/content/segmentedAudio'

# Get the list of .wav files in the directory
audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]

audio_path = os.path.join(audio_dir, audio_files[1])
# audio_path = '/content/segmentedAudio/trelis_voice_7.wav'
display(Audio(audio_path))

## THIS CAN TAKE A MINUTE TO RUN.

## Calculating Total Audio Duration
Let's calculate the total duration of all WAV files to get an idea of the dataset size.

In [16]:
import os
import wave

# Hardcoded values
sampling_rate = 24000  # Hz
frame_length = 1200  # samples per frame

def calculate_wav_duration_and_longest_snippet(folder_path):
    total_duration = 0
    max_duration_seconds = 0
    max_duration_frames = 0
    max_duration_file = ""

    for filename in os.listdir(folder_path):
        if filename.endswith('.wav'):
            with wave.open(os.path.join(folder_path, filename), 'rb') as wav_file:
                frames = wav_file.getnframes()
                rate = wav_file.getframerate()
                duration_seconds = frames / float(rate)

                # Update total duration
                total_duration += duration_seconds

                # Check if this is the longest snippet
                if duration_seconds > max_duration_seconds:
                    max_duration_seconds = duration_seconds
                    max_duration_frames = (duration_seconds * sampling_rate) / frame_length
                    max_duration_file = filename

    total_duration_minutes = total_duration / 60  # Convert total duration to minutes
    return total_duration_minutes, max_duration_seconds, max_duration_frames, max_duration_file

folder_path = './paddedAudio'
total_minutes, max_seconds, max_frames, longest_file = calculate_wav_duration_and_longest_snippet(folder_path)

print(f"Total duration of WAV files: {total_minutes:.2f} minutes")
print(f"The longest snippet is {longest_file}")
print(f"Length in seconds: {max_seconds:.2f} seconds")
print(f"Length in frames: {max_frames:.2f} frames")

# ```
# Total duration of WAV files: 19.75 minutes
# The longest snippet is trelistopten_12.wav
# Length in seconds: 29.09 seconds
# Length in frames: 581.88 frames
# ```

Total duration of WAV files: 23.43 minutes
The longest snippet is toptentips-text_10.wav
Length in seconds: 122.90 seconds
Length in frames: 2458.04 frames


!During fine-tuning, you should try to set your max frame length to a value longer than the max frames shown above !

## Create dataset

In [17]:
import os
from tqdm import tqdm
import re

language = "en-us"
input_file = "./trainingdata/output.txt"
train_output_file = "./trainingdata/train_list.txt"
val_output_file = "./trainingdata/val_list.txt"

# Read input file
with open(input_file, "r") as f:
    lines = f.readlines()

# Check number of lines read from input file
print(f"Number of lines read from input file: {len(lines)}")

# Parse lines into filenames, transcriptions, and speakers
filenames, transcriptions, speakers = zip(
    *[line.strip().split("|") for line in lines]
)

# Directly use raw transcriptions
raw_transcriptions = transcriptions

# Check number of raw transcriptions
print(f"Number of raw transcriptions: {len(raw_transcriptions)}")

# Combine filenames, raw transcriptions, and speakers into lines
raw_lines = []
for i in tqdm(range(len(filenames))):
    token_count = count_tokens(transcriptions[i])
    print(f"File: {filenames[i]}, Token length: {token_count}")
    raw_lines.append(
        (filenames[i], f"{filenames[i]}|{raw_transcriptions[i]}|{speakers[i]}\n", token_count)
    )

# Function to extract numbers from filenames for sorting
def extract_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else filename

# Sort raw lines by filename numbers
raw_lines.sort(key=lambda x: extract_number(x[0]))

# Split into training and validation sets (90% training, 10% validation)
split_index = int(len(raw_lines) * 0.9)
train_lines = raw_lines[:split_index]
val_lines = raw_lines[split_index:]

# Write training and validation lines to respective files
with open(train_output_file, "w+", encoding="utf-8") as f:
    f.writelines([line for _, line, _ in train_lines])

with open(val_output_file, "w+", encoding="utf-8") as f:
    f.writelines([line for _, line, _ in val_lines])

# Check number of lines written to output files
with open(train_output_file, "r", encoding="utf-8") as f:
    train_lines_written = f.readlines()

with open(val_output_file, "r", encoding="utf-8") as f:
    val_lines_written = f.readlines()

print(f"\nNumber of lines written to train_list.txt: {len(train_lines_written)}")
print(f"Number of lines written to val_list.txt: {len(val_lines_written)}")

print("\nToken length statistics:")
print(f"Minimum token length: {min(line[2] for line in raw_lines)}")
print(f"Maximum token length: {max(line[2] for line in raw_lines)}")
print(f"Average token length: {sum(line[2] for line in raw_lines) / len(raw_lines):.2f}")

print("\nData splitting completed.")


Number of lines read from input file: 13
Number of raw transcriptions: 13


100%|██████████| 13/13 [00:00<00:00, 82.74it/s]

File: toptentips-text_0.wav, Token length: 413
File: toptentips-text_1.wav, Token length: 410
File: toptentips-text_2.wav, Token length: 430
File: toptentips-text_3.wav, Token length: 412
File: toptentips-text_4.wav, Token length: 446
File: toptentips-text_5.wav, Token length: 418
File: toptentips-text_6.wav, Token length: 427
File: toptentips-text_7.wav, Token length: 411
File: toptentips-text_8.wav, Token length: 416
File: toptentips-text_9.wav, Token length: 409
File: toptentips-text_10.wav, Token length: 457
File: toptentips-text_11.wav, Token length: 418
File: toptentips-text_12.wav, Token length: 418

Number of lines written to train_list.txt: 11
Number of lines written to val_list.txt: 2

Token length statistics:
Minimum token length: 409
Maximum token length: 457
Average token length: 421.92

Data splitting completed.


# Uploading Audio Dataset to Hugging Face Hub
Finally, we'll upload our processed dataset to the Hugging Face Hub. Make sure you have a Hugging Face account and have logged in.

In [18]:
!pip install huggingface_hub -qU
from huggingface_hub import login
login()

In [19]:
from huggingface_hub import upload_folder, create_repo

hf_org='Trelis'
dataset_name='trelis_voice_text_512'

repo_id=f"{hf_org}/{dataset_name}"
create_repo(repo_id, repo_type="dataset") #comment out if the repo was already created

# Upload the first directory
upload_folder(
    folder_path="./paddedAudio", #or segmented if you don't run padding
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="./wavs"
)

# Upload the second directory
upload_folder(
    folder_path="./trainingdata",
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="./"
)

toptentips-text_0.wav:   0%|          | 0.00/19.8M [00:00<?, ?B/s]

toptentips-text_1.wav:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

toptentips-text_10.wav:   0%|          | 0.00/23.6M [00:00<?, ?B/s]

toptentips-text_11.wav:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Upload 13 LFS files:   0%|          | 0/13 [00:00<?, ?it/s]

...

toptentips-text_6.wav:   0%|          | 0.00/21.1M [00:00<?, ?B/s]

toptentips-text_7.wav:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

toptentips-text_8.wav:   0%|          | 0.00/19.4M [00:00<?, ?B/s]

toptentips-text_9.wav:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Trelis/trelis_voice_text_512/commit/5ad1b5797cafc24ba288069ced891ae13c8d26be', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5ad1b5797cafc24ba288069ced891ae13c8d26be', pr_url=None, pr_revision=None, pr_num=None)

## Conclusion
You have now successfully created and uploaded an audio dataset for machine learning tasks. This dataset includes segmented audio files and phonemized text. You can now use this dataset for various audio-related machine learning projects, such as text-to-speech or speech recognition.

To continue with text to speech training or fine-tuning, head back to the Trelis ADVANCED transcription repo (available for purchase on trelis.com/advanced-transcription).